In [46]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [47]:
load_dotenv()

True

In [48]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

In [49]:
class sentimentSchema(BaseModel):

    sentiment: Literal['possitive','negative'] = Field(description="Sentiment to the review")


In [50]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [51]:
structural_model = model.with_structured_output(sentimentSchema)
structural_model2 = model.with_structured_output(DiagnosisSchema)

In [52]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: dict
    response: str

In [53]:
def find_sentiment(state: ReviewState) -> Literal['positive', 'negative']:

    prompt = f"For the following review find out the sentiment {state['review']}"

    sentiment = structural_model.invoke(prompt).sentiment

    return {"sentiment" : sentiment}


def check_sentiment(state: ReviewState) -> Literal["positive_responce", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_responce'
    else: 
        return 'run_diagnosis'


def positive_responce(state: ReviewState):

    prompt = f'''Write a warm thank-you message in response to this review:\n\n"{state['review']}"\n\nAlso, kindly ask the user to leave feedback on our website.'''

    response = model.invoke(prompt).content

    return {"response" : response}


def run_diagnosis(state: ReviewState):

    prompt = f'''Diagnose this negative review:\n\n{state['review']}\n" 
                "Return issue type, tone, and urgency.'''

    output = structural_model2.invoke(prompt)

    return {"diagnosis": output.model_dump()}


def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f'''You are a support assistant.
            The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as
            '{diagnosis['urgency']}'.
            Write an empathetic, helpful resolution message.'''

    response = model.invoke(prompt).content

    return {"response": response}

In [54]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("positive_responce", positive_responce)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)
graph.add_edge("positive_responce", END)
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("negative_response", END)

workflow = graph.compile()


In [ ]:
initial_value = {"review" : "I've been using this app for about a month now, and I must say, the user interface is incredibly clean and intuitive.Everything is exactly where you'd expect it to be. It's rare to find something that just works without needing a tutorial. Great job to the design team!"}

workflow.invoke(initial_value)


{'review': "I've been using this app for about a month now, and I must say, the user interface is incredibly clean and intuitive.Everything is exactly where you'd expect it to be. It's rare to find something that just works without needing a tutorial. Great job to the design team!",
 'sentiment': 'possitive',
 'diagnosis': {'issue_type': 'Other', 'tone': 'calm', 'urgency': 'low'},
 'response': 'Hi there,\n\nThank you for reaching out and letting us know about the issue you’re experiencing. I understand that things aren’t working quite right, and I’m here to help you get it sorted out.\n\nTo make sure we can resolve this as quickly and smoothly as possible, could you please provide a little more detail about what’s happening? For example:\n\n* A brief description of the problem (what you expected to happen vs. what actually happened)\n* Any error messages or codes you’ve seen\n* The device, browser, or app version you’re using\n* Steps you’ve already tried to fix it, if any\n\nOnce we h

: 